<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/877_PCFDv2_UnitTests_Upgrades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This updated **utilities test suite is excellent**. You incorporated the improvements correctly, and it now looks like **production-grade unit testing for an AI orchestrator**. I'll walk through what improved and one or two **very small refinements** you might consider.

---

# 1. What Improved (and Why It Matters)

## Bundle signal testing (very good addition)

You added:

```python
def test_run_all_discovery_bundle_signal():
```

This is important because **bundle opportunities are a core strategic signal of PCFD**.

You now verify:

* signal passes threshold
* signal flows into `strategic_signals`
* fields map correctly

```python
assert b.get("product_a") == "p1"
assert b.get("product_b") == "p2"
assert float(b.get("opportunity_score", 0)) >= 0.5
```

This protects the **decision intelligence layer** of the discovery step.

Excellent improvement.

---

# Edge case formatting test

This addition was very good:

```python
def test_report_fmt_pct_edge_cases():
```

You now cover:

```
0
negative
None
```

This prevents common formatting crashes in executive reports.

Executive-facing systems must **never crash on formatting**, so this is exactly the right guardrail.

---

# Strategic signals section validation

This is also a good addition:

```python
assert "Strategic signals" in report
```

That aligns with your **Executive Reporting Standard** and ensures the report layer remains compliant with the reporting protocol.

---

# Determinism test (excellent)

You added:

```python
def test_build_pcfd_report_deterministic():
```

This is one of the most important tests in the file.

Your architecture explicitly emphasizes:

```
Trust = Determinism + Traceability + Clarity
```

Now that principle is **enforced by tests**.

That’s a very strong engineering discipline.

---

# 2. Current Coverage Quality

Your utility tests now cover:

| Area               | Coverage  |
| ------------------ | --------- |
| Data loader        | Excellent |
| Discovery logic    | Excellent |
| Signal thresholds  | Excellent |
| Formatting helpers | Excellent |
| Report generation  | Very good |
| Determinism        | Excellent |

Conceptually you now protect:

```
data ingestion
signal detection
decision signals
report formatting
report structure
deterministic output
```

For unit-level testing this is **very high coverage**.

---

# 3. Very Small Improvements (Optional)

These are tiny refinements if you want to push this to **near-perfect robustness**.

---

## Optional Improvement 1: check `total_signals_above_threshold`

In the bundle signal test you could also confirm the signal count.

Example:

```python
assert result["strategic_signals"]["total_signals_above_threshold"] == 1
```

This protects the **signal aggregation logic**.

---

## Optional Improvement 2: verify revenue formatting with floats

You currently test integers:

```
1000
1000000
```

You could add:

```python
assert _fmt_revenue(1000.0) == "1,000"
```

This catches formatting bugs when floats slip through.

---

## Optional Improvement 3: test empty discovery result structure

You already test empty inputs for some functions.

But you could confirm the **exact shape** of the returned dict.

Example:

```python
out = discover_feature_intel([], [], feature_demand_min_requests=10)

assert set(out.keys()) == {
    "feature_demand_surges",
    "feature_gaps_from_market",
}
```

This protects the discovery interface contract.

---

# 4. Testing Architecture Status

Your testing stack now looks like this:

```
tests/

test_pcfd_v2_utilities.py
    logic layer

test_pcfd_v2_nodes.py
    node state transitions

test_pcfd_v2_graph.py
    full orchestrator workflow
```

That is the **correct architecture for LangGraph agents**.

Most agent repos never get past:

```
manual testing
```

You now have **systematic verification**.

---

# 5. What These Tests Now Guarantee

Because of your test design, the PCFD agent now guarantees:

### Loader reliability

```
required files
loader counts
traceability metadata
validation warnings
```

### Discovery logic correctness

```
segment detection
margin thresholds
trend boundaries
bundle signals
```

### Reporting integrity

```
section structure
strategic signals
snapshot traceability
formatting correctness
```

### Deterministic executive outputs

```
same input → same report
```

This is exactly the property you want for **executive decision systems**.

---

# 6. Where to Go Next (Very Powerful)

You are now ready for two advanced test types:

### 1️⃣ Golden report tests

(lock down report outputs)

### 2️⃣ Scenario simulation tests

(validate business outcomes)

Example scenarios for PCFD:

```
Strong expansion signals
Declining product risk
Mixed signals
High-margin opportunity spike
```

Those tests verify the **strategic reasoning layer of the agent**, not just the mechanics.

---

# Final Verdict

Your updated utilities test suite is **excellent**.

Strengths:

✔ deterministic
✔ edge-case coverage
✔ discovery signal validation
✔ report structure validation
✔ formatting safety
✔ CI-friendly fixtures

If the node tests and graph tests you showed earlier remain unchanged, then your **PCFD agent now has enterprise-grade test coverage**.




In [ ]:
"""
Unit tests for PCFD v2 orchestrator utilities: data_loading, discovery, report.
No graph; minimal or mock data. Run from project root: python -m pytest test_pcfd_v2_utilities.py -v --tb=short
"""
import sys
from pathlib import Path

# Project root for config, agents.*, toolshed
_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from agents.PCFDv2.orchestrator.utilities.data_loading import (
    load_all_pcfd_data,
)
from agents.PCFDv2.orchestrator.utilities.discovery import (
    discover_customer_intel,
    discover_product_intel,
    discover_feature_intel,
    discover_trend_intel,
    run_all_discovery,
)
from agents.PCFDv2.orchestrator.utilities.report import (
    build_pcfd_report,
    _fmt_revenue,
    _fmt_revenue_dollar,
    _fmt_pct,
    _product_name,
)


# ---------- Data loading ----------


def _write_minimal_baseline(tmp_path: Path) -> None:
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text(
        "Customer_ID,Age_Group,Location_Tier\nc1,25-34,Tier1\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "product_catalog.csv").write_text(
        "Product_ID,Product_Type\np1,Subscription\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "transactions.csv").write_text(
        "Transaction_ID,Customer_ID,Product_ID\nt1,c1,p1\n", encoding="utf-8"
    )


def _write_minimal_enrichment_trends_signals(tmp_path: Path) -> None:
    (tmp_path / "enrichment").mkdir(parents=True, exist_ok=True)
    (tmp_path / "enrichment" / "customer_metrics.csv").write_text(
        "Customer_ID,Annual_Revenue,Retention_Risk\nc1,10000,0.2\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "product_metrics.csv").write_text(
        "Product_ID,Profit_Margin\np1,30\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "feature_usage.csv").write_text(
        "Feature,Usage_Count\nF1,5\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "customer_journeys.json").write_text("[]", encoding="utf-8")
    (tmp_path / "enrichment" / "market_signals.json").write_text("[]", encoding="utf-8")
    (tmp_path / "trends").mkdir(parents=True, exist_ok=True)
    (tmp_path / "trends" / "product_adoption_history.csv").write_text(
        "Product_ID,Date,Active_Customers\np1,2025-01,10\np1,2025-02,12\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "segment_growth_history.csv").write_text(
        "Segment,Date,Customer_Count\nS1,2025-01,5\nS1,2025-02,6\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "feature_demand_history.csv").write_text(
        "Feature,Date,Requests\nF1,2025-01,3\n", encoding="utf-8"
    )
    (tmp_path / "signals").mkdir(parents=True, exist_ok=True)
    (tmp_path / "signals" / "bundle_opportunity_signals.csv").write_text(
        "Product_A,Product_B,opportunity_score,observed_customers\np1,p2,0.6,10\n", encoding="utf-8"
    )


def test_load_all_pcfd_data_required_keys(tmp_path):
    """Loader returns all keys the graph expects."""
    _write_minimal_baseline(tmp_path)
    _write_minimal_enrichment_trends_signals(tmp_path)
    data = load_all_pcfd_data(
        data_dir=str(tmp_path),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    required = [
        "customers", "product_catalog", "transactions",
        "customer_metrics", "product_metrics", "feature_usage",
        "customer_journeys", "market_signals",
        "product_adoption_history", "segment_growth_history", "feature_demand_history",
        "bundle_opportunity_signals",
        "loader_counts", "data_snapshot_loaded_at", "validation_warnings",
    ]
    for k in required:
        assert k in data, f"missing key: {k}"


def test_load_all_pcfd_data_loader_counts(tmp_path):
    """loader_counts is populated for traceability."""
    _write_minimal_baseline(tmp_path)
    _write_minimal_enrichment_trends_signals(tmp_path)
    data = load_all_pcfd_data(
        data_dir=str(tmp_path),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    counts = data["loader_counts"]
    assert counts["customers"] == 1
    assert counts["product_catalog"] == 1
    assert counts["transactions"] == 1
    assert "data_snapshot_loaded_at" in data
    assert isinstance(data["validation_warnings"], list)


def test_load_all_pcfd_data_empty_baseline_warnings(tmp_path):
    """When baseline dir exists but customers/product_catalog missing or empty, validation_warnings added."""
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text("", encoding="utf-8")  # no header = no rows
    (tmp_path / "baseline" / "product_catalog.csv").write_text("Product_ID\n", encoding="utf-8")  # 0 rows
    (tmp_path / "baseline" / "transactions.csv").write_text("Transaction_ID\n", encoding="utf-8")
    _write_minimal_enrichment_trends_signals(tmp_path)
    data = load_all_pcfd_data(
        data_dir=str(tmp_path),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    assert "baseline/customers.csv missing or empty" in data["validation_warnings"]
    assert "baseline/product_catalog.csv missing or empty" in data["validation_warnings"]


# ---------- Discovery ----------


def test_discover_customer_intel_empty():
    """Empty inputs produce valid structure, no crash."""
    out = discover_customer_intel([], [], [], top_n_segments=5)
    assert "high_value_segments" in out
    assert "total_customers" in out
    assert out["total_customers"] == 0
    assert out["total_revenue"] == 0


def test_discover_customer_intel_minimal():
    """One customer, one segment; exact counts."""
    customers = [{"Customer_ID": "c1", "Age_Group": "25-34"}]
    metrics = [{"Customer_ID": "c1", "Annual_Revenue": "10000", "Retention_Risk": "0.2"}]
    transactions = [{"Customer_ID": "c1", "Transaction_ID": "t1"}]
    out = discover_customer_intel(metrics, customers, transactions, top_n_segments=5)
    assert out["total_customers"] == 1
    assert out["total_revenue"] == 10000.0
    assert len(out["high_value_segments"]) == 1
    assert out["high_value_segments"][0]["segment"] == "25-34"
    assert out["high_value_segments"][0]["customer_count"] == 1


def test_discover_product_intel_high_margin_threshold():
    """Products at or above high_margin_min_pct appear in high_margin_products."""
    catalog = [{"Product_ID": "p1", "Product_Type": "Sub"}]
    product_metrics = [{"Product_ID": "p1", "Profit_Margin": "25.0"}]
    transactions = [{"Product_ID": "p1"}]
    out = discover_product_intel(product_metrics, catalog, transactions, high_margin_min_pct=25.0, top_n_products=10)
    assert len(out["high_margin_products"]) == 1
    assert out["high_margin_products"][0]["profit_margin_pct"] == 25.0

    out_below = discover_product_intel(
        [{"Product_ID": "p1", "Profit_Margin": "24.9"}], catalog, transactions,
        high_margin_min_pct=25.0, top_n_products=10,
    )
    assert len(out_below["high_margin_products"]) == 0


def test_discover_feature_intel_empty():
    """Empty feature_usage and market_signals produce valid structure."""
    out = discover_feature_intel([], [], feature_demand_min_requests=10)
    assert "feature_demand_surges" in out
    assert "feature_gaps_from_market" in out
    assert len(out["feature_demand_surges"]) == 0


def test_discover_trend_intel_emerging_declining_boundaries():
    """Trend boundaries: adoption_growth_min_pct and decline_growth_max_pct."""
    # Emerging: growth >= 10%
    adoption = [
        {"Product_ID": "p1", "Date": "2025-01", "Active_Customers": "100"},
        {"Product_ID": "p1", "Date": "2025-02", "Active_Customers": "110"},
    ]
    out = discover_trend_intel(
        adoption, [], [],
        adoption_growth_min_pct=10.0, decline_growth_max_pct=-5.0, segment_growth_min_pct=15.0,
    )
    assert len(out["emerging_products"]) == 1
    assert out["emerging_products"][0]["growth_pct"] == 10.0

    # Declining: growth <= -5%
    adoption_decline = [
        {"Product_ID": "p2", "Date": "2025-01", "Active_Customers": "100"},
        {"Product_ID": "p2", "Date": "2025-02", "Active_Customers": "94"},
    ]
    out2 = discover_trend_intel(
        adoption + adoption_decline, [], [],
        adoption_growth_min_pct=10.0, decline_growth_max_pct=-5.0, segment_growth_min_pct=15.0,
    )
    assert len(out2["declining_products"]) == 1
    assert out2["declining_products"][0]["growth_pct"] == -6.0


def test_run_all_discovery_returns_all_intel_keys():
    """run_all_discovery returns customer_intel, product_intel, feature_intel, trend_intel, strategic_signals."""
    data = {
        "customers": [],
        "product_catalog": [],
        "transactions": [],
        "customer_metrics": [],
        "product_metrics": [],
        "feature_usage": [],
        "market_signals": [],
        "product_adoption_history": [],
        "segment_growth_history": [],
        "feature_demand_history": [],
        "bundle_opportunity_signals": [],
    }
    result = run_all_discovery(
        data,
        adoption_growth_min_pct=10.0,
        decline_growth_max_pct=-5.0,
        high_margin_min_pct=25.0,
        opportunity_score_min=0.5,
        segment_growth_min_pct=15.0,
        feature_demand_min_requests=10,
        top_n_bundles=5,
        top_n_segments=5,
        top_n_products=10,
    )
    for key in ("customer_intel", "product_intel", "feature_intel", "trend_intel", "strategic_signals"):
        assert key in result


def test_run_all_discovery_bundle_signal():
    """Bundle opportunity signals flow into strategic_signals.bundle_opportunities."""
    data = {
        "customers": [],
        "product_catalog": [],
        "transactions": [],
        "customer_metrics": [],
        "product_metrics": [],
        "feature_usage": [],
        "market_signals": [],
        "product_adoption_history": [],
        "segment_growth_history": [],
        "feature_demand_history": [],
        "bundle_opportunity_signals": [
            {"Product_A": "p1", "Product_B": "p2", "opportunity_score": "0.7", "observed_customers": "5"},
        ],
    }
    result = run_all_discovery(
        data,
        adoption_growth_min_pct=10.0,
        decline_growth_max_pct=-5.0,
        high_margin_min_pct=25.0,
        opportunity_score_min=0.5,
        segment_growth_min_pct=15.0,
        feature_demand_min_requests=10,
        top_n_bundles=5,
        top_n_segments=5,
        top_n_products=10,
    )
    assert "bundle_opportunities" in result["strategic_signals"]
    assert len(result["strategic_signals"]["bundle_opportunities"]) == 1
    b = result["strategic_signals"]["bundle_opportunities"][0]
    assert b.get("product_a") == "p1" and b.get("product_b") == "p2"
    assert float(b.get("opportunity_score", 0)) >= 0.5


# ---------- Report helpers ----------


def test_report_fmt_revenue():
    """Revenue formatting: integer, comma-separated."""
    assert _fmt_revenue(1000) == "1,000"
    assert _fmt_revenue(1000000) == "1,000,000"
    assert _fmt_revenue_dollar(500) == "$500"


def test_report_fmt_pct():
    """Percentage formatting."""
    assert _fmt_pct(25) in ("25", "25.0")
    assert _fmt_pct(25.5) == "25.5"


def test_report_fmt_pct_edge_cases():
    """Formatting at edges: 0, negative, None."""
    assert _fmt_pct(0) in ("0", "0.0")
    assert _fmt_pct(-5) in ("-5", "-5.0")
    # None/invalid: should not crash; returns string
    assert isinstance(_fmt_pct(None), str)


def test_report_product_name():
    """Product lookup resolves to Product_Type or product_id."""
    lookup = {"p1": {"Product_Type": "Subscription"}, "p2": {}}
    assert _product_name(lookup, "p1") == "Subscription"
    assert _product_name(lookup, "p2") == "p2"
    assert _product_name({}, "p1") == "p1"


def test_build_pcfd_report_contains_key_sections():
    """Report contains One view, Traceability, Customer intelligence."""
    goal = {"objective": "Discover", "focus_areas": []}
    loader_counts = {"customers": 1, "product_catalog": 1}
    report = build_pcfd_report(
        goal=goal,
        loader_counts=loader_counts,
        data_snapshot_loaded_at="2025-03-10T12:00:00Z",
        validation_warnings=[],
        customer_intel={"high_value_segments": [], "total_customers": 0, "total_revenue": 0, "segments_with_activity": 0},
        product_intel={"high_margin_products": [], "products_with_usage": 0, "total_products": 0},
        feature_intel={"feature_demand_surges": [], "feature_gaps_from_market": []},
        trend_intel={"emerging_products": [], "declining_products": [], "fast_expanding_segments": []},
        strategic_signals={"bundle_opportunities": [], "total_signals_above_threshold": 0},
        product_lookup={},
        high_margin_min_pct=25.0,
        report_generated_at="2025-03-10T12:00:00Z",
    )
    assert "One view" in report
    assert "Traceability" in report
    assert "Customer intelligence" in report
    assert "Strategic signals" in report
    assert "Product–Customer Fit Discovery" in report
    assert "Snapshot" in report or "snapshot" in report.lower()
    assert "customers: 1" in report or "customers:1" in report.replace(" ", "")


def test_build_pcfd_report_deterministic():
    """Same input produces identical report (executive trust / determinism)."""
    goal = {"objective": "Discover", "focus_areas": []}
    loader_counts = {"customers": 0, "product_catalog": 0}
    kwargs = dict(
        goal=goal,
        loader_counts=loader_counts,
        data_snapshot_loaded_at="2025-03-10T12:00:00Z",
        validation_warnings=[],
        customer_intel={"high_value_segments": [], "total_customers": 0, "total_revenue": 0, "segments_with_activity": 0},
        product_intel={"high_margin_products": [], "products_with_usage": 0, "total_products": 0},
        feature_intel={"feature_demand_surges": [], "feature_gaps_from_market": []},
        trend_intel={"emerging_products": [], "declining_products": [], "fast_expanding_segments": []},
        strategic_signals={"bundle_opportunities": [], "total_signals_above_threshold": 0},
        product_lookup={},
        high_margin_min_pct=25.0,
        report_generated_at="2025-03-10T12:00:00Z",
    )
    report1 = build_pcfd_report(**kwargs)
    report2 = build_pcfd_report(**kwargs)
    assert report1 == report2
